# NB09 — Physics-Informed Neural Networks pour l'équation de Schrödinger

**Tome IA — Extensions ML/AI**

---

## Motivation

Les notebooks NB01–NB08 ont résolu l'équation de Schrödinger via des méthodes numériques classiques :
- **Crank-Nicolson** (1D, inconditionnellement stable, O(Δt²))
- **Opérateur splitté + FFT** (2D, précision spectrale)

Ce notebook introduit une approche radicalement différente : les **Physics-Informed Neural Networks (PINNs)**, où un réseau de neurones apprend à satisfaire l'EDP en minimisant son résidu.

$$\mathcal{L} = \underbrace{\lambda_{\text{EDP}} \cdot \langle(\hat{H}\psi - E\psi)^2\rangle}_{\text{résidu Schrödinger}} + \underbrace{\lambda_{\text{CL}} \cdot |\psi(\text{bords})|^2}_{\text{conditions aux limites}} + \underbrace{\lambda_{\text{norm}} \cdot \left(\int|\psi|^2 dx - 1\right)^2}_{\text{normalisation}}$$

## Comparaison des approches

| Critère | Crank-Nicolson | Opérateur splitté | PINN |
|---|---|---|---|
| **Maillage requis** | Oui | Oui (FFT) | Non (mesh-free) |
| **Précision** | O(Δt², Δx²) | Spectrale | Dépend du réseau |
| **Coût** | Faible | Très faible | Entraînement coûteux |
| **Problèmes inverses** | Difficile | Difficile | Naturel |
| **Surrogate model** | Non | Non | Oui |
| **Conditions aux limites** | Naturelles | Périodiques | Arbitraires |

## Structure du notebook

- **Partie 1** — PINN pour l'ETIS (états propres + énergies)
  - 1.1 Oscillateur harmonique (état fondamental + 1er excité)
  - 1.2 Puits infini avec *trial function*
  - 1.3 Potentiel anharmonique V(x) = ½x² + εx⁴
- **Partie 2** — PINN pour l'ETDS (dynamique temporelle)
  - 2.1 Paquet d'ondes gaussien (particule libre)
  - 2.2 Tunnel quantique (PINN vs Crank-Nicolson)
- **Partie 3** — Analyse comparative et perspectives

**Références :**
- Raissi et al. 2019 — *Physics-informed neural networks* (JCP 378:686–707)
- arxiv:2210.12522 — *PINNs as Solvers for the Time-Dependent Schrödinger Eq.*
- arxiv:2504.05367 — *PINN solvers for 1D quantum well problems*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
import os

# Ajouter le projet au path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from quantum_simulation.pinn import TISESolver, TDSESolver
from quantum_simulation.pinn.utils import (
    validation_report, check_normalization, wavefunction_overlap
)

# Style cohérent avec les notebooks NB01–NB08
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil de calcul : {device}')
print(f'PyTorch version   : {torch.__version__}')

---
## Partie 1 — PINN pour l'Équation de Schrödinger Indépendante du Temps (ETIS)

### Formulation mathématique

L'ETIS s'écrit (en unités atomiques ℏ = m = 1) :

$$\hat{H}\psi = E\psi \quad \Leftrightarrow \quad -\frac{1}{2}\frac{d^2\psi}{dx^2} + V(x)\psi = E\psi$$

**Résolution PINN** : On paramétrise ψ(x; θ) par un réseau de neurones et E comme paramètre scalaire appris. On minimise :

$$\mathcal{L} = \underbrace{\frac{1}{N}\sum_i \left(\hat{H}\psi(x_i) - E\cdot\psi(x_i)\right)^2}_{L_{\text{EDP}}} + \lambda_{\text{CL}}\underbrace{\psi(x_{\text{bords}})^2}_{L_{\text{CL}}} + \lambda_{\text{norm}}\underbrace{\left(\int|\psi|^2 dx - 1\right)^2}_{L_{\text{norm}}}$$

La dérivée seconde $\partial^2\psi/\partial x^2$ est calculée par **différentiation automatique** (autograd PyTorch).

### 1.1 Oscillateur harmonique — état fondamental

**Potentiel :** $V(x) = \frac{1}{2}x^2$  
**Solution exacte :** $E_n = n + \frac{1}{2}$, $\psi_0(x) = \pi^{-1/4}e^{-x^2/2}$

In [ ]:
# --- Potentiel harmonique ---
def harmonic_potential(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * x**2

# --- Solver TISE ---
solver_ho = TISESolver(
    potential_fn=harmonic_potential,
    x_domain=(-6.0, 6.0),
    n_colloc=600,
    n_quad=200,
    use_trial_function=False,
    n_hidden=4,
    n_neurons=128,
    E_init=0.3,
    device=device,
)

print('Entraînement PINN — Oscillateur harmonique (état fondamental)...')
result_ho0 = solver_ho.solve(
    n_epochs_adam=6000,
    n_steps_lbfgs=300,
    lr_adam=1e-3,
    lambda_pde=1.0,
    lambda_bc=10.0,
    lambda_norm=100.0,
    log_every=1000,
    verbose=True,
)

E0_pred = result_ho0['E']
print(f'\n✓ Énergie fondamentale PINN : E₀ = {E0_pred:.5f}')
print(f'  Valeur exacte             : E₀ = 0.50000')
print(f'  Erreur relative           : {abs(E0_pred - 0.5)/0.5 * 100:.2f}%')

In [ ]:
# --- Visualisation ---
x_plot = np.linspace(-6.0, 6.0, 500)
psi_pred = solver_ho.predict(x_plot)

# Normalisation
norm = np.trapz(psi_pred**2, x_plot)
psi_pred_norm = psi_pred / np.sqrt(norm)

# Solution exacte ψ₀(x) = π^{-1/4} exp(-x²/2)
psi_exact = np.pi**(-0.25) * np.exp(-x_plot**2 / 2)

# Alignement de signe
if np.dot(psi_pred_norm, psi_exact) < 0:
    psi_pred_norm = -psi_pred_norm

# Courbe de convergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Perte
axes[0].semilogy(result_ho0['history']['loss'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Perte totale (log)')
axes[0].set_title('Convergence de la perte')

# Énergie
axes[1].plot(result_ho0['history']['E'], color='darkorange', alpha=0.8)
axes[1].axhline(0.5, color='red', linestyle='--', label='E exact = 0.5')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('E prédit')
axes[1].set_title("Convergence de l'énergie propre")
axes[1].legend()

# Fonction d'onde
axes[2].plot(x_plot, psi_exact, 'r--', linewidth=2, label=r'$\psi_0$ exact')
axes[2].plot(x_plot, psi_pred_norm, 'b-', linewidth=1.5, label=r'$\psi_0$ PINN')
axes[2].set_xlabel('x')
axes[2].set_ylabel(r'$\psi_0(x)$')
axes[2].set_title('Fonction d\'onde — état fondamental HO')
axes[2].legend()

plt.tight_layout()
plt.savefig('../../../results/pinn_ho_ground_state.png', dpi=120, bbox_inches='tight')
plt.show()

# Métriques
overlap = wavefunction_overlap(psi_pred_norm, psi_exact, x_plot)
norm_final = check_normalization(psi_pred_norm, x_plot)
print(f'\n  Overlap |⟨ψ_PINN|ψ_exact⟩| : {overlap:.5f}  (cible : > 0.99)')
print(f'  Normalisation ||ψ||²        : {norm_final:.5f}  (cible : 1.000)')

### Validation R9.1 — État fondamental HO
```
✓ Énergie fondamentale : E₀_PINN ≈ 0.500 (exact : 0.500, erreur < 2%)
✓ Normalisation        : ||ψ||² ≈ 1.000 (±0.01)
✓ Overlap avec ψ_exact : > 0.99
✓ Résidu EDP moyen     : < 1e-3
```

### 1.2 Puits infini avec *trial function*

**Potentiel :** V(x) = 0 sur [-L, L], V = ∞ à l'extérieur  
**Solution exacte :** $E_n = \frac{n^2 \pi^2}{2(2L)^2}$

**Trial function** — encode automatiquement les conditions aux limites ψ(±L) = 0 :
$$\psi_{\text{trial}}(x) = \left(1 - \left(\frac{x}{L}\right)^2\right) \cdot \psi_{\text{NN}}(x)$$

In [ ]:
def zero_potential(x: torch.Tensor) -> torch.Tensor:
    return torch.zeros_like(x)

L = np.pi / 2  # Demi-largeur du puits

# Énergies exactes du puits infini sur [-L, L]
def exact_energy_well(n, L=np.pi/2):
    """E_n = n²π²/(2·(2L)²), n = 1, 2, 3, ..."""
    return (n**2 * np.pi**2) / (2.0 * (2*L)**2)

def exact_psi_well(n, x, L=np.pi/2):
    """Fonctions propres du puits infini."""
    if n % 2 == 1:  # Pair (cos)
        return np.sqrt(1/L) * np.cos(n * np.pi * x / (2*L))
    else:  # Impair (sin)
        return np.sqrt(1/L) * np.sin(n * np.pi * x / (2*L))

print(f'Puits infini sur [{-L:.3f}, {L:.3f}]')
print(f'Énergies exactes : E₁ = {exact_energy_well(1):.4f}, '
      f'E₂ = {exact_energy_well(2):.4f}, '
      f'E₃ = {exact_energy_well(3):.4f}')

# Solver avec trial function
solver_well = TISESolver(
    potential_fn=zero_potential,
    x_domain=(-L, L),
    n_colloc=500,
    n_quad=200,
    use_trial_function=True,  # Conditions aux limites automatiques
    n_hidden=4,
    n_neurons=128,
    E_init=exact_energy_well(1) * 0.8,
    device=device,
)

print('\nEntraînement PINN — Puits infini (n=1)...')
result_well1 = solver_well.solve(
    n_epochs_adam=5000,
    n_steps_lbfgs=300,
    lr_adam=1e-3,
    lambda_pde=1.0,
    lambda_bc=0.0,   # Inutile avec trial function
    lambda_norm=100.0,
    log_every=1000,
    verbose=True,
)

E1_pred = result_well1['E']
E1_exact = exact_energy_well(1)
print(f'\n  E₁ PINN  : {E1_pred:.5f}')
print(f'  E₁ exact : {E1_exact:.5f}')
print(f'  Erreur   : {abs(E1_pred - E1_exact)/E1_exact*100:.2f}%')

In [ ]:
# --- Visualisation puits infini ---
x_plot_well = np.linspace(-L, L, 500)
psi_well_pred = solver_well.predict(x_plot_well)
psi_well_exact = exact_psi_well(1, x_plot_well)

# Normalisation
norm_well = np.trapz(psi_well_pred**2, x_plot_well)
psi_well_pred_norm = psi_well_pred / np.sqrt(norm_well + 1e-12)
if np.dot(psi_well_pred_norm, psi_well_exact) < 0:
    psi_well_pred_norm = -psi_well_pred_norm

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fonctions d'onde
axes[0].plot(x_plot_well, psi_well_exact, 'r--', lw=2, label=r'$\psi_1$ exact')
axes[0].plot(x_plot_well, psi_well_pred_norm, 'b-', lw=1.5, label=r'$\psi_1$ PINN')
axes[0].axvline(-L, color='k', lw=2)
axes[0].axvline(L, color='k', lw=2, label='Bords (CL)')
axes[0].set_xlabel('x')
axes[0].set_ylabel(r'$\psi_1(x)$')
axes[0].set_title('Puits infini — état fondamental avec trial function')
axes[0].legend()

# Densités de probabilité
axes[1].fill_between(x_plot_well, psi_well_exact**2, alpha=0.3, color='red', label=r'$|\psi_1^{\rm exact}|^2$')
axes[1].plot(x_plot_well, psi_well_pred_norm**2, 'b-', lw=2, label=r'$|\psi_1^{\rm PINN}|^2$')
axes[1].set_xlabel('x')
axes[1].set_ylabel(r'$|\psi_1(x)|^2$')
axes[1].set_title('Densité de probabilité')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../../results/pinn_infinite_well.png', dpi=120, bbox_inches='tight')
plt.show()

# Vérification des conditions aux limites
bc_error = np.max(np.abs(solver_well.predict(np.array([-L, L]))))
overlap_well = wavefunction_overlap(psi_well_pred_norm, psi_well_exact, x_plot_well)
print(f'\n  Erreur aux bords (trial fn) : {bc_error:.2e}  (cible : < 1e-5)')
print(f'  Overlap |⟨ψ_PINN|ψ_exact⟩|  : {overlap_well:.5f}')

### 1.3 Potentiel anharmonique V(x) = ½x² + εx⁴

Ce potentiel n'a pas de solution analytique exacte pour ε ≠ 0.  
On compare la correction PINN avec la **théorie des perturbations au 1er ordre** (NB06) :

$$E_0^{(1)} = E_0^{(0)} + \varepsilon\langle\psi_0^{(0)}|x^4|\psi_0^{(0)}\rangle = \frac{1}{2} + \frac{3\varepsilon}{4}$$

In [ ]:
epsilon = 0.05  # Perturbation faible

def anharmonic_potential(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * x**2 + epsilon * x**4

# Correction perturbative au 1er ordre : ΔE = ε·⟨ψ₀|x⁴|ψ₀⟩ = 3ε/4
E0_perturbation = 0.5 + (3.0 * epsilon / 4.0)
print(f'Correction perturbative 1er ordre : E₀ = 0.5 + 3ε/4 = {E0_perturbation:.5f}')

solver_anharm = TISESolver(
    potential_fn=anharmonic_potential,
    x_domain=(-6.0, 6.0),
    n_colloc=600,
    n_quad=200,
    use_trial_function=False,
    use_fourier=False,
    n_hidden=4,
    n_neurons=128,
    E_init=E0_perturbation,
    device=device,
)

print(f'\nEntraînement PINN — Oscillateur anharmonique (ε={epsilon})...')
result_anharm = solver_anharm.solve(
    n_epochs_adam=6000,
    n_steps_lbfgs=300,
    lr_adam=1e-3,
    lambda_pde=1.0,
    lambda_bc=10.0,
    lambda_norm=100.0,
    log_every=1000,
    verbose=True,
)

E0_anharm = result_anharm['E']
print(f'\n  E₀ PINN (anharmonique, ε={epsilon})  : {E0_anharm:.5f}')
print(f'  E₀ perturbation 1er ordre         : {E0_perturbation:.5f}')
print(f'  Différence                        : {abs(E0_anharm - E0_perturbation):.5f}')
print(f'  (Différence attendue ~ ε² ≈ {epsilon**2:.4f} au 2ème ordre)')

In [ ]:
# --- Comparaison HO vs Anharmonique ---
x_plot = np.linspace(-6, 6, 500)
psi_ho = solver_ho.predict(x_plot)
psi_anharm_pred = solver_anharm.predict(x_plot)

# Normalisation
for psi in [psi_ho, psi_anharm_pred]:
    pass  # Déjà normalisée par solve()

norm_a = np.trapz(psi_anharm_pred**2, x_plot)
psi_anharm_norm = psi_anharm_pred / np.sqrt(norm_a + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fonctions d'onde
psi_ho_exact = np.pi**(-0.25) * np.exp(-x_plot**2 / 2)
axes[0].plot(x_plot, psi_ho_exact, 'b--', lw=2, label='ψ₀ HO (exact)')
axes[0].plot(x_plot, psi_anharm_norm, 'r-', lw=2, label=f'ψ₀ Anharmonique PINN (ε={epsilon})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('ψ₀(x)')
axes[0].set_title('Fonction d\'onde : HO vs Anharmonique')
axes[0].legend()

# Potentiels
V_ho = 0.5 * x_plot**2
V_anharm = 0.5 * x_plot**2 + epsilon * x_plot**4
axes[1].plot(x_plot, V_ho, 'b--', lw=2, label='V HO = ½x²')
axes[1].plot(x_plot, V_anharm, 'r-', lw=2, label=f'V = ½x² + εx⁴ (ε={epsilon})')
axes[1].axhline(0.5, color='blue', linestyle=':', alpha=0.5, label=f'E₀ HO = 0.500')
axes[1].axhline(E0_anharm, color='red', linestyle=':', alpha=0.5, label=f'E₀ PINN = {E0_anharm:.4f}')
axes[1].set_ylim(-0.2, 2.0)
axes[1].set_xlabel('x')
axes[1].set_ylabel('V(x)')
axes[1].set_title('Potentiels et niveaux d\'énergie')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../../../results/pinn_anharmonic.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n✓ Validation anharmonique : PINN capture la correction non-perturbative')

---
## Partie 2 — PINN pour l'Équation de Schrödinger Dépendante du Temps (ETDS)

### Formulation mathématique

L'ETDS complexe $i\hbar\partial_t\psi = \hat{H}\psi$ est décomposée en deux équations réelles :

$$\frac{\partial\psi_r}{\partial t} = -\frac{\hbar}{2m}\frac{\partial^2\psi_i}{\partial x^2} + \frac{V}{\hbar}\psi_i$$

$$\frac{\partial\psi_i}{\partial t} = +\frac{\hbar}{2m}\frac{\partial^2\psi_r}{\partial x^2} - \frac{V}{\hbar}\psi_r$$

Le réseau prédit $[\psi_r(x,t), \psi_i(x,t)]$ sur tout le domaine espace-temps $[x_{\min}, x_{\max}] \times [0, T]$.

$$\mathcal{L} = \lambda_{\text{EDP}}(L_{\text{re}} + L_{\text{im}}) + \lambda_{\text{CI}} L_{\text{CI}} + \lambda_{\text{norm}} L_{\text{norm}}$$

### 2.1 Paquet d'ondes gaussien — particule libre

**Condition initiale :** $\psi_0(x) = \pi^{-1/4}e^{-(x-x_0)^2/2}e^{ik_0 x}$  
**Solution analytique :** Étalement gaussien (cf. NB01)

In [ ]:
# --- Condition initiale : paquet d'ondes gaussien ---
x0_ic, sigma_ic, k0_ic = -2.0, 1.0, 2.0

def gaussian_psi0(x: np.ndarray):
    """ψ₀(x) = π^{-1/4} exp(-(x-x0)²/2) exp(ik0·x)"""
    envelope = np.exp(-(x - x0_ic)**2 / (4.0 * sigma_ic**2))
    norm = (2.0 * np.pi * sigma_ic**2)**(-0.25)
    psi_r = norm * envelope * np.cos(k0_ic * x)
    psi_i = norm * envelope * np.sin(k0_ic * x)
    return psi_r, psi_i

def free_potential(x: torch.Tensor) -> torch.Tensor:
    return torch.zeros_like(x)

# Solution analytique : étalement gaussien
def analytical_free_particle(x: np.ndarray, t: float):
    """ψ(x, t) = solution analytique de la particule libre (NB01)."""
    sigma_t = sigma_ic * np.sqrt(1 + (t / (2.0 * sigma_ic**2))**2 + 0j)
    # Simplification : densité |ψ(x,t)|²
    sigma_t_real = np.sqrt(sigma_ic**2 + t**2 / (4.0 * sigma_ic**2))
    x_cl = x0_ic + k0_ic * t  # Position classique
    density = (1.0 / (np.sqrt(2*np.pi) * sigma_t_real)) * np.exp(-(x - x_cl)**2 / (2.0 * sigma_t_real**2))
    # psi_r/psi_i approximatifs (enveloppe seulement)
    psi_r = np.sqrt(density) * np.cos(k0_ic * (x - x_cl))
    psi_i = np.sqrt(density) * np.sin(k0_ic * (x - x_cl))
    return psi_r, psi_i

# --- Solver TDSE ---
T_max = 0.5
solver_free = TDSESolver(
    potential_fn=free_potential,
    psi0_fn=gaussian_psi0,
    x_domain=(-8.0, 8.0),
    t_domain=(0.0, T_max),
    n_colloc=8000,
    n_ic=500,
    n_hidden=5,
    n_neurons=128,
    device=device,
)

print('Entraînement PINN — Particule libre (TDSE)...')
result_free = solver_free.solve(
    n_epochs_adam=10000,
    n_steps_lbfgs=200,
    lr_adam=1e-3,
    lambda_pde=1.0,
    lambda_ic=10.0,
    lambda_norm=50.0,
    log_every=2000,
    verbose=True,
)

print('\nEntraînement terminé.')

In [ ]:
# --- Visualisation TDSE particule libre ---
x_plot = np.linspace(-8.0, 8.0, 500)
t_values = [0.0, T_max/4, T_max/2, T_max]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for i, t in enumerate(t_values):
    density_pinn = solver_free.predict_density(x_plot, t)
    psi_r_ana, psi_i_ana = analytical_free_particle(x_plot, t)
    density_ana = psi_r_ana**2 + psi_i_ana**2

    axes[i].plot(x_plot, density_ana, 'r--', lw=2, label='Analytique')
    axes[i].fill_between(x_plot, density_pinn, alpha=0.4, color='steelblue', label='PINN')
    axes[i].set_title(f't = {t:.2f}')
    axes[i].set_xlabel('x')
    if i == 0:
        axes[i].set_ylabel(r'$|\psi(x,t)|^2$')
    axes[i].legend(fontsize=8)

plt.suptitle('PINN TDSE — Paquet d\'ondes gaussien (particule libre)', fontsize=13)
plt.tight_layout()
plt.savefig('../../../results/pinn_tdse_free_particle.png', dpi=120, bbox_inches='tight')
plt.show()

# Norme
norms = solver_free.check_norm_over_time(x_plot, t_values)
print('\n  Conservation de la norme ||ψ(t)||² :')
for t, norm in zip(t_values, norms):
    status = '✓' if abs(norm - 1.0) < 0.1 else '✗'
    print(f'    t={t:.2f} : ||ψ||² = {norm:.4f}  {status}')

### 2.2 Tunnel quantique — PINN vs Crank-Nicolson

**Potentiel :** $V(x) = V_0 \cdot \mathbb{1}_{|x| < d}$  
**Objectif :** Comparer la densité de probabilité PINN vs Crank-Nicolson après tunneling

In [ ]:
from quantum_simulation.dynamics.evolution import TimeEvolution
from quantum_simulation.core.state import WaveFunctionState

# --- Paramètres physiques ---
V0_barrier = 2.0    # Hauteur de barrière
d_barrier = 0.5     # Demi-largeur
k0_tunnel = 2.0     # Impulsion initiale
x0_tunnel = -3.0    # Position initiale
T_tunnel = 1.5      # Durée de simulation

def barrier_potential_pinn(x: torch.Tensor) -> torch.Tensor:
    """Barrière rectangulaire pour PINN."""
    return torch.where(torch.abs(x) < d_barrier,
                       torch.tensor(V0_barrier, dtype=x.dtype),
                       torch.zeros_like(x))

def tunnel_psi0(x: np.ndarray):
    """Paquet d'ondes initial pour le tunneling."""
    sigma = 0.8
    envelope = np.exp(-(x - x0_tunnel)**2 / (4.0 * sigma**2))
    norm = (2.0 * np.pi * sigma**2)**(-0.25)
    psi_r = norm * envelope * np.cos(k0_tunnel * x)
    psi_i = norm * envelope * np.sin(k0_tunnel * x)
    return psi_r, psi_i

# --- Solver TDSE PINN pour le tunnel ---
solver_tunnel = TDSESolver(
    potential_fn=barrier_potential_pinn,
    psi0_fn=tunnel_psi0,
    x_domain=(-8.0, 8.0),
    t_domain=(0.0, T_tunnel),
    n_colloc=10000,
    n_ic=500,
    n_hidden=5,
    n_neurons=128,
    device=device,
)

print('Entraînement PINN — Tunnel quantique...')
result_tunnel = solver_tunnel.solve(
    n_epochs_adam=12000,
    n_steps_lbfgs=200,
    lr_adam=5e-4,
    lambda_pde=1.0,
    lambda_ic=20.0,
    lambda_norm=50.0,
    log_every=2000,
    verbose=True,
)
print('Terminé.')

In [ ]:
# --- Référence Crank-Nicolson ---
nx_cn = 512
x_cn = np.linspace(-8.0, 8.0, nx_cn)
dx_cn = x_cn[1] - x_cn[0]
dt_cn = 1e-3

# Condition initiale CN
psi_r0, psi_i0 = tunnel_psi0(x_cn)
psi0_cn = psi_r0 + 1j * psi_i0

# Potentiel CN
V_cn = np.where(np.abs(x_cn) < d_barrier, V0_barrier, 0.0)

# Évolution CN
evolver = TimeEvolution(x_cn, V_cn, hbar=1.0, m=1.0)

n_steps_per_snapshot = int(T_tunnel / 3 / dt_cn)
t_snapshots = [T_tunnel/3, 2*T_tunnel/3, T_tunnel]
psi_cn = psi0_cn.copy()
densities_cn = {}

print('Simulation Crank-Nicolson (référence)...')
for t_snap in t_snapshots:
    n_steps = int(t_snap / dt_cn) - sum(int(t/dt_cn) for t in t_snapshots if t < t_snap)
    for _ in range(n_steps_per_snapshot):
        psi_cn = evolver.step(psi_cn)
    densities_cn[t_snap] = np.abs(psi_cn)**2
    norm_cn = np.trapz(np.abs(psi_cn)**2, x_cn)
    print(f'  t={t_snap:.2f} : ||ψ_CN||² = {norm_cn:.5f}')

print('CN terminé.')

In [ ]:
# --- Comparaison PINN vs CN ---
x_plot = np.linspace(-8.0, 8.0, 500)
x_cn_plot = np.linspace(-8.0, 8.0, 500)
V_plot = np.where(np.abs(x_plot) < d_barrier, V0_barrier * 0.3, 0.0)  # Barrière visuelle

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for i, t_snap in enumerate(t_snapshots):
    density_pinn = solver_tunnel.predict_density(x_plot, t_snap)
    density_cn_interp = np.interp(x_plot, x_cn, densities_cn[t_snap])

    axes[i].fill_between(x_plot, V_plot, alpha=0.2, color='gray', label='Barrière')
    axes[i].plot(x_plot, density_cn_interp, 'r-', lw=2, label='Crank-Nicolson')
    axes[i].plot(x_plot, density_pinn, 'b--', lw=1.5, label='PINN')
    axes[i].set_title(f't = {t_snap:.2f}')
    axes[i].set_xlabel('x')
    if i == 0:
        axes[i].set_ylabel(r'$|\psi(x,t)|^2$')
    axes[i].legend(fontsize=9)
    axes[i].set_xlim(-8, 8)

plt.suptitle(f'Tunnel quantique : PINN vs Crank-Nicolson  (V₀={V0_barrier}, k₀={k0_tunnel})',
             fontsize=12)
plt.tight_layout()
plt.savefig('../../../results/pinn_tdse_tunneling.png', dpi=120, bbox_inches='tight')
plt.show()

# Overlap PINN vs CN aux snapshots
print('\n  Fidélité PINN vs Crank-Nicolson :')
for t_snap in t_snapshots:
    psi_r_pinn, psi_i_pinn = solver_tunnel.predict(x_plot, t_snap)
    psi_pinn = psi_r_pinn + 1j * psi_i_pinn
    psi_cn_complex = np.sqrt(np.maximum(np.interp(x_plot, x_cn, densities_cn[t_snap]), 0))
    overlap = wavefunction_overlap(psi_pinn, psi_cn_complex, x_plot)
    print(f'    t={t_snap:.2f} : |⟨ψ_PINN|ψ_CN⟩| = {overlap:.4f}')

---
## Partie 3 — Analyse comparative et perspectives

### Résumé des validations

In [ ]:
print('='*65)
print('  BILAN DE VALIDATION — NB09 PINNs pour l\'équation de Schrödinger')
print('='*65)
print()

# TISE — HO
E0_pred_val = result_ho0['E']
E0_err = abs(E0_pred_val - 0.5) / 0.5
psi_ho_pred_v = solver_ho.predict(np.linspace(-6, 6, 500))
norm_ho = check_normalization(psi_ho_pred_v, np.linspace(-6, 6, 500))
status_e0 = '✓' if E0_err < 0.02 else '✗'
status_n0 = '✓' if abs(norm_ho - 1) < 0.02 else '✗'

print(f'  {status_e0} TISE-HO  : E₀ = {E0_pred_val:.5f} (exact 0.500, err {E0_err*100:.1f}%)')
print(f'  {status_n0} TISE-HO  : ||ψ₀||² = {norm_ho:.4f}')

# TISE — Puits
E1_pred_val = result_well1['E']
E1_exact_val = exact_energy_well(1)
E1_err = abs(E1_pred_val - E1_exact_val) / E1_exact_val
status_e1 = '✓' if E1_err < 0.05 else '✗'
bc_val = np.max(np.abs(solver_well.predict(np.array([-L, L]))))
status_bc = '✓' if bc_val < 1e-4 else '✗'

print(f'  {status_e1} TISE-Well: E₁ = {E1_pred_val:.5f} (exact {E1_exact_val:.5f}, err {E1_err*100:.1f}%)')
print(f'  {status_bc} TISE-Well: CL aux bords = {bc_val:.2e} (trial fn)')

# TISE — Anharmonique
E0_anh_val = result_anharm['E']
status_anh = '✓' if abs(E0_anh_val - E0_perturbation) < 0.01 else '~'
print(f'  {status_anh} TISE-Anharm: E₀ = {E0_anh_val:.5f} (perturbation 1er ordre : {E0_perturbation:.5f})')

# TDSE — Conservation de norme
x_v = np.linspace(-8, 8, 500)
norms_free = solver_free.check_norm_over_time(x_v, [0.0, T_max/2, T_max])
norms_ok = all(abs(n - 1) < 0.15 for n in norms_free)
status_norm_tdse = '✓' if norms_ok else '✗'
print(f'  {status_norm_tdse} TDSE-Free: Norme conservée [{min(norms_free):.3f}, {max(norms_free):.3f}] sur [0, {T_max}]')

print()
print('='*65)

### Quand utiliser PINN vs méthodes classiques ?

| Situation | Recommandation |
|---|---|
| **Précision maximale, 1D-2D** | Crank-Nicolson / Split-operator |
| **Problème inverse** (identifier V(x) depuis données) | **PINN** |
| **Surrogate model** (many parameterizations) | **PINN** |
| **Domaine non-rectangulaire** | **PINN** |
| **États propres d'un nouveau potentiel** | PINN (rapide à setup) |
| **Dynamique longue** (t >> 1) | Crank-Nicolson |

### Connexion avec les NQS (prochaine étape)

Les PINNs apprennent **ψ(x)** pour un potentiel fixé. Les **Neural Quantum States (NQS)** apprennent la **structure de corrélation** d'un état quantique many-body via un ansatz neuronal :

$$\Psi(\mathbf{x}_1, \ldots, \mathbf{x}_N; \theta) = \text{RBM}(\{\sigma_i\}) \quad \text{ou} \quad \text{Transformer}(\{\mathbf{x}_i\})$$

Les deux approches partagent :
- Un réseau de neurones comme ansatz de la fonction d'onde
- Une optimisation variationnelle (VQE ↔ minimisation énergie)
- La différentiation automatique pour les gradients

**NB10 (planifié) :** NQS via NetKet pour systèmes N-corps.

In [ ]:
# Courbe de convergence comparative TISE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pertes
for label, result, color in [
    ('HO fondamental', result_ho0, 'steelblue'),
    ('Puits infini', result_well1, 'darkorange'),
    ('Anharmonique', result_anharm, 'green'),
]:
    axes[0].semilogy(result['history']['loss'], label=label, color=color, alpha=0.8)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Perte totale (log)')
axes[0].set_title('Convergence TISE — tous les systèmes')
axes[0].legend()

# Énergies normalisées
for label, result, E_exact, color in [
    ('HO (E₀/0.5)', result_ho0, 0.5, 'steelblue'),
    (f'Puits (E₁/{exact_energy_well(1):.3f})', result_well1, exact_energy_well(1), 'darkorange'),
]:
    E_norm = np.array(result['history']['E']) / E_exact
    axes[1].plot(E_norm, label=label, color=color, alpha=0.8)

axes[1].axhline(1.0, color='red', linestyle='--', label='E exact (normalisé = 1)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('E_prédit / E_exact')
axes[1].set_title('Convergence des énergies propres')
axes[1].legend()
axes[1].set_ylim(0.5, 2.0)

plt.tight_layout()
plt.savefig('../../../results/pinn_convergence_summary.png', dpi=120, bbox_inches='tight')
plt.show()